In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import matplotlib as mpl


In [ ]:
def ABS_SHAP(df_shap, df):
    shap_v = pd.DataFrame(df_shap)
    feature_list = df.columns
    shap_v.columns = feature_list
    df_v = df.copy().reset_index().drop('index',axis=1)
    
    corr_list = [np.corrcoef(shap_v[i], df_v[i])[1][0] for i in feature_list]
    corr_df = pd.concat([pd.Series(feature_list), pd.Series(corr_list)], axis=1).fillna(0)
    corr_df.columns = ['Variable','Corr']
    corr_df['Sign'] = np.where(corr_df['Corr'] > 0, 'red', 'blue')

    shap_abs = np.abs(shap_v)
    k = pd.DataFrame(shap_abs.mean()).reset_index()
    k.columns = ['Variable','SHAP_abs']
    k2 = k.merge(corr_df, on='Variable', how='inner')
    return k2

    


In [ ]:
def MEAN_SHAP_SIGNED(df_shap, df):
    shap_v = pd.DataFrame(df_shap)
    feature_list = df.columns
    shap_v.columns = feature_list
    df_v = df.copy().reset_index().drop('index', axis=1)

    corr_list = [np.corrcoef(shap_v[i], df_v[i])[1][0] for i in feature_list]
    corr_df = pd.DataFrame({'Variable': feature_list, 'Corr': corr_list}).fillna(0)

    shap_signed_mean = shap_v.mean().reset_index()
    shap_signed_mean.columns = ['Variable', 'SHAP_mean']
    
    result = shap_signed_mean.merge(corr_df, on='Variable')
    return result


In [4]:
def process_file(filename, label):
    with open(filename, 'r') as infile:
        lines = infile.readlines()

    first_line = lines[0].strip().split('\t')
    aas = ['A','C','D','E','F','G','H','I','K','L','M','N','P','Q','R','S','T','V','W','Y']
    feature_names = first_line[2:-2] + aas
    cols = feature_names + [first_line[-1]]

    all_line = []
    for line in lines[1:]:
        data = line.strip().split('\t')
        sequence = data[-2]
        s = list(sequence)
        compositions = [float(s.count(aa))/len(s) for aa in aas]
        PTMs = [float(d) for d in data[2:-2]]
        one_line = PTMs + compositions + [float(data[-1])]
        all_line.append(one_line)

    all_data = pd.DataFrame(all_line, columns=cols)
    feature_data = all_data[feature_names]
    labels = all_data.maxTP
    X_train, _, y_train, _ = train_test_split(feature_data, labels, test_size=0.2, random_state=42)

    rf = RandomForestRegressor(n_estimators=10)
    rf.fit(X_train, y_train)
    explainer = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X_train)
    shap_df = ABS_SHAP(shap_values, X_train)
    # shap_df = MEAN_SHAP_SIGNED(shap_values, X_train)
    shap_df['Label'] = label
    return shap_df


In [ ]:
def main():

    mpl.rcParams['pdf.fonttype'] = 42  
    mpl.rcParams['ps.fonttype'] = 42
    mpl.rcParams['svg.fonttype'] = 'none' 

    # Read SHAP data
    df_pp = process_file('../../Results/SHAP_Analysis/FakeTPresMore_PP.txt', 'Ppa')
    df_sce = process_file('../../Results/SHAP_Analysis/FakeTPresMore_SCE.txt', 'Sce')
    df_kmx = process_file('../../Results/SHAP_Analysis/FakeTPresMore_KMX.txt', 'Kma')

    # Sort variables
    sorted_vars = df_kmx.sort_values(by='SHAP_abs', ascending=True)['Variable'].tolist()
    all_shap = pd.concat([df_kmx, df_sce, df_pp], ignore_index=True)
    all_shap['Variable'] = pd.Categorical(all_shap['Variable'], categories=sorted_vars[::-1], ordered=True)

    # Set font
    mpl.rcParams['font.family'] = 'Arial'
    mpl.rcParams['font.size'] = 7

    # Set figure size (units: inches)
    fig, ax = plt.subplots(figsize=(3.54, 2.76), facecolor='none')

    # Set colors
    custom_palette = {
        'Kma': (0.431, 0.690, 0.584),
        'Sce': (0.784, 0.545, 0.659),
        'Ppa': (0.553, 0.737, 0.816),
    }

    # Plot bars
    sns.barplot(
        data=all_shap,
        y='SHAP_abs',
        x='Variable',
        hue='Label',
        orient='v',
        palette=custom_palette,
        ax=ax,
        dodge=True,
        width=0.8
    )

    # Remove bar edges
    for patch in ax.patches:
        patch.set_linewidth(0)
        patch.set_edgecolor('none')

    # Customize axes
    ax.spines['left'].set_visible(True)
    ax.spines['bottom'].set_visible(True)
    ax.spines['left'].set_color('black')
    ax.spines['bottom'].set_color('black')
    ax.spines['left'].set_linewidth(1)
    ax.spines['bottom'].set_linewidth(1)


    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    #   Ticks
    ax.tick_params(axis='both', which='both',
                   direction='in',   
                   length=2,           
                   width=1,
                   color='black',
                   top=False,
                   right=False)
    ax.tick_params(axis='y', which='both', left=True)


    # Y-axis ticks and labels with scientific notation
    max_y = all_shap['SHAP_abs'].max()
    exp = int(np.floor(np.log10(max_y)))
    y_ticks = np.linspace(0, max_y, num=5)
    ax.set_yticks(y_ticks)
    tick_labels = [f'{y / 10**exp:.1f}' for y in y_ticks]
    ax.set_yticklabels(tick_labels)

    mpl.rcParams['mathtext.fontset'] = 'dejavusans'  
    txt = rf'$\boldsymbol{{\times}}\ \mathbf{{10}}^{{\mathbf{{{exp}}}}}$'

    ax.text(0, 1, txt,
            transform=ax.transAxes,
            ha='right', va='bottom',
            fontsize=7, fontname='Arial',  
            )

    # Tick font bold
    for label in ax.get_xticklabels() + ax.get_yticklabels():
        label.set_fontname('Arial')
        label.set_fontsize(7)
        label.set_fontweight('bold')

    # Axis labels
    ax.set_ylabel("Mean |SHAP| Value", fontsize=7, fontname='Arial', weight='bold')
    ax.set_xlabel("Features", fontsize=7, fontname='Arial', weight='bold')

    # Rotate x-axis labels
    plt.xticks(rotation=45, ha='center', fontsize=7, fontname='Arial', weight='bold')

    # Legend
    legend = ax.legend(loc='upper right', fontsize=7, title_fontsize=7)
    legend.get_frame().set_linewidth(0)
    legend.get_frame().set_alpha(0)
    for text in legend.get_texts():
        text.set_fontname('Arial')
        text.set_fontsize(7)
        text.set_fontweight('bold')
    legend.get_title().set_fontname('Arial')
    legend.get_title().set_fontsize(7)
    legend.get_title().set_fontweight('bold')

    # X-axis margin
    ax.margins(x=0.03)
    ax.grid(False)


    plt.tight_layout()
    plt.savefig("figure.pdf", dpi=400, transparent=True, format="pdf", 
            metadata={'Creator': 'Matplotlib'})
    # plt.savefig("Fig5d.png", dpi=400, transparent=True)
    # plt.savefig("SHAPnew_horizontal.svg", format="svg", bbox_inches="tight", transparent=True)
    plt.show()



In [ ]:
if __name__ == "__main__":
    main()